# 7 — Curvas de Aprendizaje y Robustez al Tamaño del Dataset

Entrena los modelos con fracciones crecientes del dataset (5%, 10%, 25%, 50%, 75%, 100%) y mide cómo evoluciona el rendimiento.

**Preguntas que responde:**
- ¿Cuántas muestras se necesitan para que RF alcance buen MAE?
- ¿Hay sobreajuste con pocos datos?
- ¿Qué modelo converge más rápido?
- ¿Merece la pena recoger más datos o ya hemos saturado?

**Modelos:** Ridge Lineal, Random Forest, Gradient Boosting (los más representativos)

**Métricas:** MAE de regresión + Accuracy de clasificación (20 clases)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75, 1.00]
N_REPS = 3  # repeticiones por fracción (distintas semillas) para barra de error

In [ ]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

if FEAT_JSON.exists():
    with open(FEAT_JSON) as f: FEAT_COLS = json.load(f)
else:
    FEAT_COLS = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Features: {len(FEAT_COLS)}")

df_tr = df[df["split"] == "train"]
df_te = df[df["split"] == "test"]

X_full = df_tr[FEAT_COLS].values; y_reg_full = df_tr["temperature"].values
y_cls_full = df_tr["temperature"].astype(int).astype(str).values + "C"
X_te = df_te[FEAT_COLS].values
y_te_reg = df_te["temperature"].values
y_te_cls = df_te["temperature"].astype(int).astype(str).values + "C"

print(f"Train completo: {X_full.shape}  Test: {X_te.shape}")

In [ ]:
def make_models_reg():
    return {
        "Ridge": Pipeline([("imp", SimpleImputer(strategy="mean")),
                           ("sc",  StandardScaler()),
                           ("m",   Ridge(alpha=1.0))]),
        "Random Forest": Pipeline([("imp", SimpleImputer(strategy="mean")),
                                   ("m",   RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42))]),
        "Gradient Boosting": Pipeline([("imp", SimpleImputer(strategy="mean")),
                                       ("m",   GradientBoostingRegressor(n_estimators=100, learning_rate=0.05,
                                                                          max_depth=5, random_state=42))]),
    }

def make_models_cls():
    return {
        "Random Forest": Pipeline([("imp", SimpleImputer(strategy="mean")),
                                   ("m",   RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42))]),
        "Gradient Boosting": Pipeline([("imp", SimpleImputer(strategy="mean")),
                                       ("m",   GradientBoostingClassifier(n_estimators=100, learning_rate=0.05,
                                                                           max_depth=5, random_state=42))]),
    }

print("Modelos definidos")

In [ ]:
# ── Experimento: curvas de aprendizaje (regresión) ────────────────────────────
print("=" * 55)
print("REGRESIÓN — curvas de aprendizaje")
print("=" * 55)

curves_reg = {name: {"mae": [], "mae_std": [], "r2": [], "n": []} for name in make_models_reg()}

for frac in FRACTIONS:
    n = max(int(len(X_full) * frac), 50)
    mae_per_rep = {name: [] for name in make_models_reg()}
    for seed in range(N_REPS):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X_full), n, replace=False)
        X_sub = X_full[idx]; y_sub = y_reg_full[idx]
        for name, model in make_models_reg().items():
            model.fit(X_sub, y_sub)
            mae = mean_absolute_error(y_te_reg, model.predict(X_te))
            mae_per_rep[name].append(mae)
    for name in make_models_reg():
        vals = mae_per_rep[name]
        curves_reg[name]["mae"].append(np.mean(vals))
        curves_reg[name]["mae_std"].append(np.std(vals))
        curves_reg[name]["n"].append(n)
    print(f"  frac={frac:.0%} n={n:5d}  " +
          "  ".join(f"{nm}={np.mean(mae_per_rep[nm]):.3f}°C" for nm in make_models_reg()))

print("Curvas de regresión OK")

In [ ]:
# ── Experimento: curvas de aprendizaje (clasificación) ───────────────────────
print("=" * 55)
print("CLASIFICACIÓN — curvas de aprendizaje")
print("=" * 55)

curves_cls = {name: {"acc": [], "acc_std": [], "n": []} for name in make_models_cls()}

for frac in FRACTIONS:
    n = max(int(len(X_full) * frac), 50)
    acc_per_rep = {name: [] for name in make_models_cls()}
    for seed in range(N_REPS):
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X_full), n, replace=False)
        X_sub = X_full[idx]; y_sub = y_cls_full[idx]
        for name, model in make_models_cls().items():
            model.fit(X_sub, y_sub)
            acc = accuracy_score(y_te_cls, model.predict(X_te))
            acc_per_rep[name].append(acc)
    for name in make_models_cls():
        vals = acc_per_rep[name]
        curves_cls[name]["acc"].append(np.mean(vals)*100)
        curves_cls[name]["acc_std"].append(np.std(vals)*100)
        curves_cls[name]["n"].append(n)
    print(f"  frac={frac:.0%} n={n:5d}  " +
          "  ".join(f"{nm}={np.mean(acc_per_rep[nm])*100:.2f}%" for nm in make_models_cls()))

print("Curvas de clasificación OK")

In [ ]:
# ── Figura: curvas de aprendizaje ─────────────────────────────────────────────
colors = {"Ridge": "#ED7D31", "Random Forest": "#2E75B6", "Gradient Boosting": "#70AD47"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regresión — MAE
ax = axes[0]
for name, data in curves_reg.items():
    ns = data["n"]
    maes = data["mae"]
    stds = data["mae_std"]
    ax.plot(ns, maes, "o-", color=colors[name], label=name, linewidth=2, markersize=6)
    ax.fill_between(ns,
                    [m-s for m,s in zip(maes,stds)],
                    [m+s for m,s in zip(maes,stds)],
                    color=colors[name], alpha=0.15)
ax.set_xlabel("Nº muestras de entrenamiento", fontsize=10)
ax.set_ylabel("MAE [°C] en test", fontsize=10)
ax.set_title("Curvas de aprendizaje — Regresión", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xscale("log")

# Clasificación — Accuracy
ax = axes[1]
for name, data in curves_cls.items():
    ns = data["n"]
    accs = data["acc"]
    stds = data["acc_std"]
    ax.plot(ns, accs, "o-", color=colors[name], label=name, linewidth=2, markersize=6)
    ax.fill_between(ns,
                    [a-s for a,s in zip(accs,stds)],
                    [a+s for a,s in zip(accs,stds)],
                    color=colors[name], alpha=0.15)
ax.set_xlabel("Nº muestras de entrenamiento", fontsize=10)
ax.set_ylabel("Accuracy [%] en test", fontsize=10)
ax.set_title("Curvas de aprendizaje — Clasificación (20 clases)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xscale("log")

plt.suptitle("¿Cuántos datos necesita cada modelo?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("curvas_aprendizaje.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Punto de saturación: ¿dónde se estabiliza el MAE? ─────────────────────────
print("Análisis de saturación (regresión — Random Forest):")
rf_data = curves_reg["Random Forest"]
for n, mae, std in zip(rf_data["n"], rf_data["mae"], rf_data["mae_std"]):
    print(f"  n={n:6d}  MAE={mae:.3f} ± {std:.3f} °C")

# Mejora marginal al doblar datos
maes = rf_data["mae"]
ns   = rf_data["n"]
print("\nMejora marginal al pasar de una fracción a la siguiente:")
for i in range(1, len(maes)):
    delta = maes[i-1] - maes[i]
    pct   = delta / maes[i-1] * 100
    print(f"  {ns[i-1]:6d} → {ns[i]:6d}  ΔMAE={delta:.3f}°C  ({pct:.1f}% mejora)")

In [ ]:
# ── Figura: tabla resumen final ───────────────────────────────────────────────
rows = []
for frac_i, frac in enumerate(FRACTIONS):
    row = {"Fracción": f"{frac:.0%}"}
    for name in curves_reg:
        row[f"MAE {name} [°C]"] = round(curves_reg[name]["mae"][frac_i], 3)
    for name in curves_cls:
        row[f"Acc {name} [%]"] = round(curves_cls[name]["acc"][frac_i], 2)
    rows.append(row)
df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))